# Spark Cluster Configuration — All Parameters

This notebook displays **all available** Spark parameters including those
not explicitly set (showing their default or runtime values).

> ⚠️ Spark has thousands of parameters. This notebook groups them by prefix
> and provides filtering and search.

**Notebook 1** (`01_active_configuration`) shows only the explicitly configured
parameters with explanations. Use this notebook for exploration and debugging.


In [ ]:
from pyspark.sql import SparkSession
import pandas as pd

MASTER = 'spark://spark-master:7077'
spark = (SparkSession.builder
    .appName('cluster-config-all')
    .master(MASTER)
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
# Re-suppress noisy loggers after setLogLevel resets them (Spark 4.x / log4j2)
_jvm = spark.sparkContext._jvm
_ctx = _jvm.org.apache.logging.log4j.LogManager.getContext(False)
_cfg = _ctx.getConfiguration()
for _logger_name in [
    'org.apache.iceberg.hadoop.HadoopTableOperations',
    'org.apache.hadoop.util.NativeCodeLoader',
]:
    _lc = _jvm.org.apache.logging.log4j.core.config.LoggerConfig(_logger_name,
              _jvm.org.apache.logging.log4j.Level.ERROR, False)
    _cfg.addLogger(_logger_name, _lc)
_ctx.updateLoggers()
print(f"Spark {spark.version}")


In [ ]:
# Load ALL parameters from the running SparkSession
all_conf = spark.sparkContext.getConf().getAll()
print(f"Total parameters loaded: {len(all_conf)}")

df_all = pd.DataFrame(all_conf, columns=['Parameter', 'Value'])
df_all = df_all.sort_values('Parameter').reset_index(drop=True)
df_all['Prefix'] = df_all['Parameter'].str.split('.').str[:3].str.join('.')

print("\nTop 20 most common prefixes:")
print(df_all['Prefix'].value_counts().head(20).to_string())


In [ ]:
# ── Filter by prefix ────────────────────────────────────────────────────────
# Change prefix_filter to explore a specific category:
#   'spark.sql'            — all SQL parameters
#   'spark.executor'       — executor settings
#   'spark.driver'         — driver settings
#   'spark.sql.adaptive'   — AQE only
#   'spark.sql.catalog'    — catalog settings
#   'spark.shuffle'        — shuffle settings
#   ''                     — ALL parameters

prefix_filter = 'spark.sql'   # ← change here

if prefix_filter:
    filtered = df_all[df_all['Parameter'].str.startswith(prefix_filter)]
else:
    filtered = df_all

print(f"Parameters with prefix '{prefix_filter}': {len(filtered)}")
print()

try:
    from IPython.display import display, HTML
    display(HTML(filtered[['Parameter','Value']].to_html(index=False, border=1,
        justify='left').replace('<th>', '<th style="text-align:left;padding:4px 8px;background:#2c3e50;color:white">').replace('<td>', '<td style="padding:4px 8px;border-bottom:1px solid #ddd;font-size:13px">')))
except ImportError:
    pd.set_option('display.max_colwidth', 80)
    print(filtered[['Parameter','Value']].to_string(index=False))


In [ ]:
# ── Explicitly set vs default ────────────────────────────────────────────────
explicitly_set = {
    'spark.master', 'spark.executor.memory', 'spark.driver.memory',
    'spark.executor.cores', 'spark.sql.shuffle.partitions',
    'spark.serializer', 'spark.sql.adaptive.enabled',
    'spark.sql.adaptive.coalescePartitions.enabled', 'spark.eventLog.enabled',
    'spark.eventLog.dir', 'spark.history.fs.logDirectory',
    'spark.history.ui.port', 'spark.sql.extensions',
    'spark.sql.catalog.local', 'spark.sql.catalog.local.type',
    'spark.sql.catalog.local.warehouse', 'spark.sql.catalog.spark_catalog',
    'spark.sql.execution.arrow.pyspark.enabled', 'spark.sql.ansi.enabled',
    'spark.driver.extraJavaOptions', 'spark.executor.extraJavaOptions',
    'spark.shuffle.sort.bypassMergeThreshold',
}

df_all['Explicitly Set'] = df_all['Parameter'].isin(explicitly_set).map(
    {True: '✅ Yes', False: '— Default'})

print("Explicitly set vs default:")
print(df_all['Explicitly Set'].value_counts().to_string())
print()

set_params = df_all[df_all['Explicitly Set']=='✅ Yes'][['Parameter','Value','Explicitly Set']]
try:
    from IPython.display import display, HTML
    display(HTML(set_params.to_html(index=False, border=1,
        justify='left').replace('<th>', '<th style="text-align:left;padding:4px 8px;background:#27ae60;color:white">').replace('<td>', '<td style="padding:4px 8px;border-bottom:1px solid #ddd">')))
except ImportError:
    print(set_params.to_string(index=False))


In [ ]:
# ── Search parameters by keyword ─────────────────────────────────────────────
# Change search_term to find relevant parameters:
search_term = 'memory'   # ← change here

results = df_all[
    df_all['Parameter'].str.contains(search_term, case=False) |
    df_all['Value'].str.contains(search_term, case=False)
]
print(f"Parameters containing '{search_term}': {len(results)}")
print()
try:
    from IPython.display import display, HTML
    display(HTML(results[['Parameter','Value']].to_html(index=False, border=1,
        justify='left').replace('<th>', '<th style="text-align:left;padding:4px 8px;background:#8e44ad;color:white">').replace('<td>', '<td style="padding:4px 8px;border-bottom:1px solid #ddd">')))
except ImportError:
    print(results[['Parameter','Value']].to_string(index=False))


In [ ]:
# ── Export to CSV ────────────────────────────────────────────────────────────
import pathlib
out = '/workspace/data/spark_all_config.csv'
df_all[['Parameter','Value','Explicitly Set']].to_csv(out, index=False)
print(f"Saved to: {out}")
print(f"Total parameters: {len(df_all)}")
